In [15]:
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision.transforms import v2
from torchinfo import summary
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report
from sklearn.preprocessing import label_binarize
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import KFold
from PIL import Image
from scipy import signal
import pandas as pd
import numpy as np
import warnings
import math
import time
import os
warnings.filterwarnings('ignore')

In [16]:
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 150)

In [17]:
while not os.path.isdir(os.path.join(os.getcwd(), 'data')):
    os.chdir("../") # set cwd to root dir

In [18]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [19]:
if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))

NVIDIA GeForce RTX 3050 Ti Laptop GPU


In [20]:
writer = SummaryWriter()

In [21]:
class ButterworthFilter(object):
    def __init__(self, cutoff, order, fs):
        self.cutoff = cutoff
        self.order = order
        self.fs = fs

    def __call__(self, input):
        if torch.is_tensor(input):
            input = input.numpy() # convert to numpy array before using signal package

        nyquist = 0.5 * self.fs
        normal_cutoff = self.cutoff / nyquist
        b, a = signal.butter(self.order, normal_cutoff, btype='low', analog=False)
        smoothed_imu_signal = signal.filtfilt(b, a, input)
        
        return torch.from_numpy(smoothed_imu_signal.copy()) # convert back to tensor

In [22]:
class PadTrimToLength(object):
    def __init__(self, padlen):
        self.padlen = padlen

    def __call__(self, input):
        pad = nn.ZeroPad1d((0, max(0, self.padlen - input.shape[1])))(input)
        output = torch.narrow(pad, 1, 0, self.padlen)
        assert output.shape[1] == self.padlen, "Not equal to length of pad"
        return output

In [23]:
class Normalize(object):
    def __init__(self, mean, std, epsilon=1e-7):
        self.mean = mean
        self.std = std
        self.epsilon = epsilon

    def __call__(self, input):
        output = (input - self.mean) / (self.std + self.epsilon)
        return output

In [24]:
class DUO_GAIT(Dataset):
    def __init__(self, allowed_sensors, participant_id, remove_outliers=True, transform=None, target_transform=None):
        self.participant_id = participant_id
        self.allowed_sensors = allowed_sensors
        self.num_channels = len(allowed_sensors) * 6

        lf_control_df = pd.read_csv(f"data/DUO-GAIT/processed/OG_st_control/sub_{participant_id:02}/left_foot_core_params.csv")
        lf_control_df.rename({ "timestamps": "start_times" }, axis=1, inplace=True)
        lf_control_df['is_fatigue'] = 0
        lf_control_df['is_right_foot'] = 0
        lf_control_df['Participant'] = participant_id

        self.create_start_end_samples_strides(lf_control_df, is_control=1)

        rf_control_df = pd.read_csv(f"data/DUO-GAIT/processed/OG_st_control/sub_{participant_id:02}/right_foot_core_params.csv")
        rf_control_df.rename({ "timestamps": "start_times" }, axis=1, inplace=True)
        rf_control_df['is_fatigue'] = 0
        rf_control_df['is_right_foot'] = 1
        rf_control_df['Participant'] = participant_id

        self.create_start_end_samples_strides(rf_control_df, is_control=1)

        lf_fatigue_df = pd.read_csv(f"data/DUO-GAIT/processed/OG_st_fatigue/sub_{participant_id:02}/left_foot_core_params.csv")
        lf_fatigue_df.rename({ "timestamps": "start_times" }, axis=1, inplace=True)
        lf_fatigue_df['is_fatigue'] = 1
        lf_fatigue_df['is_right_foot'] = 0
        lf_fatigue_df['Participant'] = participant_id

        self.create_start_end_samples_strides(lf_fatigue_df, is_control=0)

        rf_fatigue_df = pd.read_csv(f"data/DUO-GAIT/processed/OG_st_fatigue/sub_{participant_id:02}/right_foot_core_params.csv")
        rf_fatigue_df.rename({ "timestamps": "start_times" }, axis=1, inplace=True)
        rf_fatigue_df['is_fatigue'] = 1
        rf_fatigue_df['is_right_foot'] = 1
        rf_fatigue_df['Participant'] = participant_id

        self.create_start_end_samples_strides(rf_fatigue_df, is_control=0)

        self.foot_strides_df = pd.concat([lf_control_df, rf_control_df, lf_fatigue_df, rf_fatigue_df], axis=0)

        if remove_outliers:
            self.foot_strides_df = self.foot_strides_df[self.foot_strides_df['is_outlier']==False].reset_index(drop=True)

        self.transform = transform
        self.target_transform = target_transform

    def create_start_end_samples_strides(self, df, is_control):
        df.sort_values(by='stride_index', inplace=True)
        df['start_samples'] = df['ic_samples'].shift(1)
        target_time = df.loc[0, 'start_times']
        
        protocol = "control" if is_control else "fatigue"

        fatigue_df = pd.read_csv(f"data/DUO-GAIT/interim/OG_st_{protocol}/sub_{self.participant_id:02}/LF.csv")
        fatigue_df.rename({ "timestamp": "Time (secs)", "Unnamed: 0": "Sample" }, axis=1, inplace=True)
        fatigue_df['Delta (secs)'] = fatigue_df['Time (secs)'] - fatigue_df['Time (secs)'].min() # delta time

        ts_eq_check = fatigue_df['Delta (secs)'].apply(lambda x: math.isclose(x, target_time, rel_tol=1e-5))
        start_sample = fatigue_df[ts_eq_check]['Sample'].item() - fatigue_df['Sample'].min()

        df.loc[0, 'start_samples'] = start_sample
        df['start_samples'] = df['start_samples'] + fatigue_df['Sample'].min()
        df['end_samples'] = df['ic_samples'] + fatigue_df['Sample'].min() - 1 # make it inclusive for ending samples too

        df['start_samples'] = df['start_samples'].astype(np.int64)
        df['end_samples'] = df['end_samples'].astype(np.int64)

    def __len__(self):
        return len(self.foot_strides_df)

    def __getitem__(self, idx):
        row = self.foot_strides_df.iloc[idx]

        imu_signals_df = pd.DataFrame()

        for sensor_location in self.allowed_sensors:
            protocol = "fatigue" if row['is_fatigue'] else "control"
            sensor_df = pd.read_csv(f"data/DUO-GAIT/interim/OG_st_{protocol}/sub_{self.participant_id:02}/{sensor_location}.csv")
            sensor_df.rename({ "timestamp": "Time (secs)", "Unnamed: 0": "Sample" }, axis=1, inplace=True)
            sensor_df = sensor_df[(sensor_df['Sample'] >= row['start_samples']) & (sensor_df['Sample'] <= row['end_samples'])].reset_index(drop=True)

            for sensor_type in ["Gyr", "Acc"]:
                for direction in ["X", "Y", "Z"]:
                    col_name = f"{sensor_type}{direction}"
                    new_col_name = f"{sensor_location}_{col_name}"

                    imu_signals_df[new_col_name] = sensor_df[col_name]

        imu_signals_np = imu_signals_df.to_numpy().transpose()
        label = row["is_fatigue"]

        if self.transform:
            imu_signals_np = self.transform(imu_signals_np)

        if self.target_transform:
            label = self.target_transform(label)

        return imu_signals_np, label

In [25]:
transform = v2.Compose([
    ButterworthFilter(cutoff=10.0, order=3, fs=128.0),
    PadTrimToLength(padlen=150),
])

In [26]:
num_participants = 18
ignore_participant_ids = [4,7,16]
allowed_sensors = ["LL"]
num_stride_samples = 150

In [27]:
start_epoch = 0
k_folds = 5
batch_size = 32
epochs = 100 # TODO - change this to infinite and implement early stopping instead
learning_rate = 0.02
learning_rate_factor = 0.5
train_percent = 0.90
early_stopping_rounds = 3

In [ ]:
for pid in range(1, num_participants+1):
    if pid in ignore_participant_ids: # skip participants that are messy and have missing info
        continue
    
    dataset = DUO_GAIT(allowed_sensors=allowed_sensors, participant_id=pid, transform=transform)
    dataset_size = len(dataset)
    num_channels = dataset.num_channels

    train_size = int(dataset_size * train_percent)
    test_size = dataset_size - train_size
    # 90% for k-fold cv; 10% for final unbiased evaluation

    cvset, test_set = random_split(dataset, [train_size, test_size])

    kf = KFold(n_splits=k_folds, shuffle=True, random_state=42) # 5 fold cv
    for fold, (train_idx, valid_idx) in enumerate(kf.split(cvset)):
        train_set = Subset(cvset, train_idx)
        valid_set = Subset(cvset, valid_idx)

        train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
        valid_loader = DataLoader(valid_set, batch_size=batch_size, shuffle=True)

        dataset.transform = transform # reset transform for normalization

        full_train_data = []
        for batch_idx, (train_features, train_labels) in enumerate(train_loader):
            full_train_data.append(train_features)

        full_train_data = torch.concat(full_train_data, dim=0)
        std, mean = torch.std_mean(full_train_data, dim=(0, 2), keepdim=True)
        # compute mean and std over all channels separately

        std_squeeze = torch.squeeze(std, dim=0)
        mean_squeeze = torch.squeeze(mean, dim=0)

        dataset.transform = v2.Compose([
            ButterworthFilter(cutoff=10.0, order=3, fs=128.0),
            PadTrimToLength(padlen=150),
            Normalize(mean=mean, std=std)
        ]) # update transform to normalize batches

        for epoch in range(start_epoch, epochs):
            train_epoch_loss, val_epoch_loss = 0.0, 0.0

            train_epoch_preds, train_epoch_labels = [], []
            val_epoch_preds, val_epoch_labels = [], []

            train_start_time = time.time()
            model.train()
            for batch_idx, (train_features, train_labels) in enumerate(train_loader):
                batch_size = len(train_labels)

                train_features = train_features.to(device)
                train_labels = train_labels.to(device) # move data to device

                optimizer.zero_grad()

                logits = model(train_features)

tensor([[[[ 0.7915,  0.8525,  0.9040,  ...,  0.7452,  1.0158,  0.0348],
          [-1.8061, -2.1939, -2.4876,  ...,  0.2841, -0.0086, -0.0949],
          [-0.8122, -0.7171, -0.6333,  ..., -0.7019, -0.8746,  0.0343],
          [-1.4754, -0.4982,  0.3917,  ..., -0.4801, -0.9328,  0.6129],
          [ 0.2791,  0.5933,  0.8837,  ..., -0.4840, -0.4837, -2.0368],
          [-0.0734, -0.2499, -0.3938,  ...,  0.7550,  0.3224,  0.7370]]],


        [[[ 0.8786,  0.9143,  0.9518,  ...,  0.0348,  0.0348,  0.0348],
          [ 0.5779,  0.6504,  0.7293,  ..., -0.0949, -0.0949, -0.0949],
          [-0.9124, -0.9417, -0.9750,  ...,  0.0343,  0.0343,  0.0343],
          [ 0.3293,  0.2074,  0.1368,  ...,  0.6129,  0.6129,  0.6129],
          [ 0.6587,  0.7138,  0.7980,  ..., -2.0368, -2.0368, -2.0368],
          [ 0.4855,  0.5703,  0.6549,  ...,  0.7370,  0.7370,  0.7370]]],


        [[[ 0.3872,  0.3232,  0.2714,  ..., -1.2742, -1.0144, -0.7482],
          [-0.7267, -1.1135, -1.4395,  ..., -0.3239, -0.

KeyboardInterrupt: 